In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.stats import sigma_clipped_stats

from sklearn.cluster import DBSCAN

from Kakapo.photometry import forced_photometry

%matplotlib widget

In [2]:
def initial_filter(df):
    df = df[(df.fwhm > 0.8) & (df.snr >= 3) & 
            (df.psfdiff <= 2) & (df.poisson_thresh >= 0.75) & 
            (abs(df.correlation) >= 0.05)]
    
    return df

def _grouping(corr: pd.DataFrame, f_dist: int = 48) -> pd.DataFrame | None:
    """
    Group detections with DBSCAN in O(N log N) time, using only C/Fortran
    code paths from scikit-learn (no Python callback per point pair).
    """
    if corr.empty:
        return None

    # Scale the frame axis so that `eps` of 1.25 encloses ±f_dist frames.
    data = corr[['xcentroid', 'ycentroid', 'frame']].values.astype(np.float32)
    data[:, 2] *= 1.5 / f_dist

    db = DBSCAN(eps=1.5,
                min_samples=5,
                metric='euclidean',           # now fully compiled
                algorithm='auto',        # fastest for 3‑D Euclidean
                n_jobs=1)                     # keep it serial – you already parallelise at a higher level
    labels = db.fit_predict(data)

    corr = corr.assign(cluster=labels)
    corr = corr[corr.cluster != -1]           # drop noise points

    return corr if not corr.empty else None

# def mask_detections(correlation, psfdiff, fwhm, snr, 
#                     roundness, poisson_thresh, xstd, ystd):
    
#     mask =  (correlation >= self.corrlim) & (psfdiff <= self.difflim) & \
#             (fwhm <= self.fwhmlim) & (fwhm >= 0.9) & \
#             (snr >= self.snrlim) & (snr < 10000) & (abs(roundness) <= self.roundness) & \
#             (poisson_thresh >= self.poiss_val) & \
#             (xstd <= self.dist_cut) & (ystd <= self.dist_cut)
    
    
#     return mask

In [3]:
file = '/Users/zgl12/Modules/Kakapo/Data/csv_files/c3/c3_t205922648.csv'
df = pd.read_csv(file)

In [4]:
df_new = df[(df['fwhm'] >= 0.8) & (df['fwhm'] <= 5) & 
   (df['roundness'] <= 0.95) & (abs(df['correlation'])>= 0.2) & 
   (abs(df['snr'])>= 3) & (abs(df['snr'])<= 1e4) & (df['psfdiff'] <= 1.2) & 
   (df['psfdiff'] <= 1.2) & (df['poisson_thresh'] >= 1)]

In [5]:
# plt.figure()
# plt.scatter(df.xcentroid.values, df.ycentroid.values, c = df.frame.values)
# plt.xlabel(r'$x$')
# plt.ylabel(r'$y$')
# plt.show()

# plt.figure()
# plt.scatter(df.fwhm.values, df.roundness.values, c = df.frame.values)
# plt.xlabel(r'FWHM')
# plt.ylabel(r'Roundness')
# plt.show()

# plt.figure()
# plt.scatter(df.snr.values, df.correlation.values, c = df.frame.values)
# plt.xlabel(r'SNR')
# plt.ylabel(r'Correlation')
# plt.show()

# plt.figure()
# plt.scatter(df.psfdiff.values, df.poisson_thresh.values, c = df.frame.values)
# plt.xlabel(r'PSF-Diff.')
# plt.ylabel(r'Poisson Threshold')
# plt.show()

In [6]:
df_1 = initial_filter(df)

df_1 = _grouping(df_1, f_dist = 48)

In [7]:
df_1

,xcentroid,ycentroid,fwhm,roundness,pa,max_value,flux,mag,snr,flux_err,...,n_detections,poisson_thresh,ref_flux,campaign,target_id,ra,dec,filename,ref_frame,cluster
416,5.685167,11.327468,0.983116,0.311234,34.034091,1.004151,8.304073,-2.298228,23.215172,0.357700,...,1,0.824851,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,0
486,6.660243,11.105325,1.542535,0.491628,4.897027,1.777637,14.711161,-2.919117,3.560447,4.131830,...,1,1.459029,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,0
498,5.816735,11.369904,1.754738,0.115857,79.117236,2.947479,19.330674,-3.215617,6.404158,3.018457,...,1,1.574247,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,0
523,5.820921,11.379757,1.213905,0.254185,86.851127,1.346270,8.526243,-2.326894,3.097822,2.752335,...,1,1.106443,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,0
554,6.520417,10.984662,1.762445,0.167550,151.397746,1.825159,9.266779,-2.417322,5.001264,1.852887,...,1,1.477374,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9349,7.372738,6.132774,1.332933,0.434682,101.740153,15.999277,77.233426,-4.719513,3.654639,21.132982,...,1,10.938976,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,19
9354,7.423664,6.043166,1.325934,0.384037,103.333951,17.079706,80.055805,-4.758482,3.759183,21.296063,...,1,11.686079,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,19
9367,7.313915,6.242240,1.334583,0.494499,98.070865,10.993680,55.606675,-4.362817,3.222030,17.258275,...,1,7.664077,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,19
9369,7.287729,6.318611,1.323293,0.579579,97.075544,12.521301,64.317480,-4.520823,3.802042,16.916563,...,1,8.561928,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,19


In [8]:
for cluster in np.unique(df_1.cluster.values):
    
    temp_df = df_1[df_1.cluster == cluster]
    print(cluster)
    print(len(temp_df), temp_df.frame.min(), temp_df.frame.max())
    x, _, xstd = sigma_clipped_stats(temp_df.xcentroid.values, sigma = 3)
    y, _, ystd = sigma_clipped_stats(temp_df.ycentroid.values, sigma = 3)
    
    print(f"{x:.2f} +/- {xstd:.2f}")
    print(f"{y:.2f} +/- {ystd:.2f}")
    
    # x, _, xstd = sigma_clipped_stats(temp_df.xcentroid.values, sigma = 3)
    # y, _, ystd = sigma_clipped_stats(temp_df.ycentroid.values, sigma = 3)
    correlation, _, _ = sigma_clipped_stats(temp_df.correlation.values, sigma = 3)
    psfdiff, _, _  = sigma_clipped_stats(temp_df.psfdiff.values, sigma = 3)
    snr, _, _  = sigma_clipped_stats(temp_df.snr.values, sigma = 3)
    fwhm, _, _  = sigma_clipped_stats(temp_df.fwhm.values, sigma = 3)
    roundness, _, _  = sigma_clipped_stats(temp_df.roundness.values, sigma = 3)
    poisson_thresh, _, _  = sigma_clipped_stats(temp_df.poisson_thresh.values, sigma = 3)

    print('Corr.', correlation)
    print('PSF Diff.', psfdiff)
    print('SNR', snr)
    print('FWHM', fwhm)
    print('Roundness', roundness)
    print('Poiss.', poisson_thresh)
    print()
    

    


0
5 130.0 160.0
6.10 +/- 0.41
11.23 +/- 0.16
Corr. 0.26421559763426805
PSF Diff. 0.590702677038559
SNR 8.255772463106748
FWHM 1.4513475806382947
Roundness 0.2680909305840089
Poiss. 1.288388835895548

1
6 173.0 225.0
4.20 +/- 0.02
7.13 +/- 0.03
Corr. 0.6271588657128034
PSF Diff. 0.5695526009155388
SNR 3.9579870681385394
FWHM 1.5526576862847774
Roundness 0.45958144099126724
Poiss. 3.3379689589077794

2
97 330.0 938.0
9.18 +/- 0.05
8.63 +/- 0.69
Corr. 0.6818469181334802
PSF Diff. 0.5847183821668005
SNR 3.6037616152190823
FWHM 1.4586139673363516
Roundness 0.5207219870592803
Poiss. 2.334172146818117

3
17 331.0 404.0
4.18 +/- 0.05
7.24 +/- 0.09
Corr. 0.6801122162321921
PSF Diff. 0.589970155017064
SNR 3.696792354475153
FWHM 1.527531288122472
Roundness 0.5236851264990277
Poiss. 3.005516915076142

4
7 331.0 434.0
10.01 +/- 0.05
6.93 +/- 0.12
Corr. 0.1432892626078271
PSF Diff. 0.6006663815116325
SNR 3.543101784130983
FWHM 1.607440853141272
Roundness 0.5119142109454482
Poiss. 1.878439973096031



In [ ]:


# mask_detections(correlation, psfdiff, fwhm, snr, 
#                     roundness, poisson_thresh, xstd, ystd)

In [ ]:
diff = np.load('/Users/zgl12/Modules/Kakapo/Data/difference_arrays/c3/diff_c3_t205922648.npy')

In [ ]:
fluxes = forced_photometry(diff, 6.36, 6.35, None)

In [ ]:
plt.figure()
plt.plot(fluxes)
plt.axvline(2695.0, color = 'r')
plt.axvline(3385.0, color = 'r')
plt.show()